In [ ]:
import os


def get_hf_token():
    """Récupère le HF_TOKEN depuis Colab userdata ou os.environ."""

    # Priorité 1 : os.environ (CLI Colab, local, CI/CD)
    token = os.environ.get("HF_TOKEN")

    if token:
        print("✅ HF_TOKEN trouvé dans os.environ")
        return token

    # Priorité 2 : Colab userdata (interface web)
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")

        if token:
            print("✅ HF_TOKEN trouvé dans userdata")
            return token

    except (ImportError, Exception):
        pass

    # Aucun token trouvé
    raise ValueError(
        "❌ HF_TOKEN introuvable.\n"
        "Solutions :\n"
        "  - CLI Colab  : colab exec --env HF_TOKEN ...\n"
        "  - Web Colab  : ajoutez le secret via l'icône 🔑\n"
        "  - Local      : export HF_TOKEN=hf_..."
    )


HF_TOKEN = get_hf_token()

# Propagation (utile si vous lancez des sous-processus)
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"   Valeur : {HF_TOKEN[:6]}...{HF_TOKEN[-4:]}")

In [ ]:
import os
import sys
from pathlib import Path

# ============================================================
# PYTHONPATH pour le projet
# ============================================================

# Après le clone, le projet sera dans /content/llm-dataset/
# Mais ici on est encore avant le clone → on prépare juste

PROJECT_ROOT = Path("/content/llm-dataset")
SRC_DIR = PROJECT_ROOT / "src"

# Ces chemins seront valides APRÈS le clone
# On les exporte en variables d'environnement pour que les
# sous-processus (!python -m ...) y aient accès

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["SRC_DIR"] = str(SRC_DIR)

# On construit un PYTHONPATH propre
existing = os.environ.get("PYTHONPATH", "")
new_path = f"{SRC_DIR}:{PROJECT_ROOT}"

if existing:
    new_path = f"{new_path}:{existing}"

os.environ["PYTHONPATH"] = new_path

print(f"✅ PYTHONPATH préparé :")
print(f"   PROJECT_ROOT = {PROJECT_ROOT}")
print(f"   SRC_DIR      = {SRC_DIR}")
print(f"   PYTHONPATH   = {os.environ['PYTHONPATH']}")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Synobotix/llm-dataset.git"
REPO_NAME = "llm-dataset"

PROJECT_DIR = Path(f"/content/{REPO_NAME}")

if PROJECT_DIR.exists():
    print(f"📁 Repo déjà cloné — pull")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    print("📥 Clonage...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)

print(f"✅ Projet : {os.getcwd()}")

In [ ]:
import os
import subprocess

subprocess.run(
    "curl -sSL https://install.python-poetry.org | python3 -",
    shell=True,
    check=True,
)

os.environ["PATH"] = (
    f"{os.path.expanduser('~')}/.local/bin:"
    + os.environ["PATH"]
)

subprocess.run(["poetry", "--version"], check=True)

In [ ]:
import subprocess
import sys

print("Python :", sys.version)
print()

# ============================================================
# 1. Mettre pip à jour
# ============================================================

subprocess.run(
    ["poetry", "run", "pip", "install", "--upgrade", "pip"],
    check=True,
)

# ============================================================
# 2. Désinstaller les versions CPU
# ============================================================

subprocess.run(
    ["poetry", "run", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio"],
    check=False,   # ne pas planter si absents
)

# ============================================================
# 3. Installer torch CUDA (versions stables épinglées)
# ============================================================

result = subprocess.run(
    ["poetry", "run", "pip", "install",
     "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    capture_output=True,
    text=True,
)

# ============================================================
# 4. Si CUDA échoue → fallback sur CPU
# ============================================================

if result.returncode != 0:

    print("⚠️ Échec de l'installation torch CUDA")
    print("STDERR :", result.stderr[-2000:])
    print()
    print("→ Tentative avec torch CPU...")

    subprocess.run(
        ["poetry", "run", "pip", "install",
         "torch", "torchvision", "torchaudio"],
        check=True,
    )

# ============================================================
# 5. Vérifier
# ============================================================

result = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import torch; "
     "print('torch', torch.__version__); "
     "print('cuda', torch.cuda.is_available()); "
     "print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("STDERR :", result.stderr[-500:])

In [ ]:
from huggingface_hub import login, whoami

login(token=HF_TOKEN, add_to_git_credential=False)
print(f"✅ HF : {whoami(token=HF_TOKEN)['name']}")

In [ ]:
import os
import subprocess

result = subprocess.run(
    ["python", "-m", "scripts.hub.hub_download"],
    env=os.environ.copy(),   # ← hérite du PYTHONPATH
    text=True,
)

if result.returncode != 0:
    print("⚠️ Cellule 6 échouée (peut être normal au premier run)")

In [ ]:
import os
import subprocess

result = subprocess.run(
    ["python", "-m", "scripts.nemotron.download_and_prepare"],
    env=os.environ.copy(),
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("Échec cellule 7")

In [ ]:
import os
import subprocess

result = subprocess.run(
    ["python", "-m", "scripts.split_dataset"],
    env=os.environ.copy(),
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("Échec cellule 8")

In [ ]:
import os
import subprocess

result = subprocess.run(
    ["python", "-m", "src.tokenizer.train_tokenizer"],
    env=os.environ.copy(),
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("Échec cellule 9")

In [ ]:
import os
import subprocess

result = subprocess.run(
    ["python", "-m", "scripts.tokenize_dataset"],
    env=os.environ.copy(),
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("Échec cellule 10")

In [ ]:
import os
import subprocess

result = subprocess.run(
    ["python", "-m", "scripts.bitnet.train_bitnet"],
    env=os.environ.copy(),
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("Échec cellule 11")

In [ ]:
import os
import subprocess

result = subprocess.run(
    ["python", "-m", "scripts.hub.sync"],
    env=os.environ.copy(),
    text=True,
)